## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# Module 11 — Learning GPU Programming with AI

## Final project: accelerate the LHCb Allen MVA trigger kernel

You may use AI as much as you like in this notebook. The point is **not** to
stop you using it — it's to make sure you still learn what the course teaches
while you do. Read [README.md](./README.md) first.

You will drive an AI assistant through a five-phase loop against a real HEP
kernel. **You own phases 1, 3, 4, and 5. The AI only owns phase 2.**

| Phase | Owner | Output |
|-------|-------|--------|
| 1. SPECIFY | you | kernel contract |
| 2. GENERATE | AI | the code |
| 3. PREDICT | you | occupancy / bound / speedup / risks, *before running* |
| 4. VERIFY | you | a test harness you wrote |
| 5. DIAGNOSE | you | profiler-backed explanation + next step |

The graded artifact is your **workbook** (copy [TEMPLATE_ai_workbook.md](./TEMPLATE_ai_workbook.md)),
not the kernel. Grading criteria are in [grading/grading_guide.md](./grading/grading_guide.md);
self-paced learners use [grading/self_check.md](./grading/self_check.md).

## 0. Environment check

Confirm you have a GPU and a working `nvcc`. If either cell fails, you are not
on the lab machine — stop and switch to the GPU environment.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!nvcc --version

## 1. Meet the baseline

[final-project/mva_infer_baseline.cu](./final-project/mva_infer_baseline.cu) is
the single-hidden-layer neural network the LHCb Allen trigger uses to classify
events (ReLU + sigmoid). It is **correct but deliberately slow**:

- one thread per event, model weights re-read from global memory by every thread,
- plain `cudaMalloc` / `cudaMemcpy`, no unified memory,
- a single synchronous kernel, no transfer/compute overlap,
- standard `expf` / division, no fast math, no constant or shared memory.

Read the source before you touch anything. You cannot supervise code you do not
understand.

Build and run it. Note the two numbers it prints: **throughput (M events/sec)**
— your metric — and **RESULT: PASS/FAIL** — your correctness gate.

In [ ]:
!nvcc -O3 -arch=native final-project/mva_infer_baseline.cu -o mva_baseline

In [ ]:
!./mva_baseline 20000000

**Record the baseline throughput in your workbook now.** Every speedup you claim
for the rest of this module is measured against this number.

## The loop

The cells below are one pass of the loop. Copy the workbook template, then work
through the phases. Repeat the loop for each optimization
(`mva_infer_v1.cu`, `v2.cu`, ...). Keep the baseline untouched as your CPU
reference.

### Phase 1 — SPECIFY (you)

Before you ask the AI for anything, write the kernel contract in your workbook:

- inputs (shape, dtype, **layout** — is the input array-of-structs or
  struct-of-arrays?),
- outputs (shape, dtype, valid range),
- the launch configuration you intend and why,
- the memory model (which arrays live in global / constant / shared / unified
  memory?),
- the single metric you'll judge success by.

Pick **one** optimization to target this pass. Suggested order (each maps to a
course concept):

1. launch configuration / grid-stride (Module 01),
2. coalesce the input layout (AoS → SoA),
3. put the model weights in `__constant__` or shared memory,
4. fast-math intrinsics — *and prove the output still passes the gate*,
5. unified memory + prefetch (Module 02),
6. streams: overlap H2D/D2H with compute (Module 03),
7. block-size / occupancy tuning from profiler evidence (Module 03).

### Phase 2 — GENERATE (AI)

Ask your AI assistant to produce the optimized variant **as a new file**, e.g.
`final-project/mva_infer_v1.cu`. Paste the prompt you used into your workbook.

Do **not** overwrite the baseline. Do not let the AI also write your test
harness — you write that in Phase 4.

When the file exists, compile it below (edit the filename).

In [ ]:
# Edit the filename to your AI-generated variant.
!nvcc -O3 -arch=native final-project/mva_infer_v1.cu -o mva_v1

### Phase 3 — PREDICT (you) — *before running*

Commit predictions to your workbook **before** you run the cell below:

- expected occupancy and launch geometry,
- memory-bound or compute-bound? (This network is tiny per event — think hard
  about which side you're on and what that implies for the ceiling.)
- expected speedup vs baseline, with a number and a reason,
- correctness risks you can already see in the generated code. Check it against
  the failure-mode list in the README: dropped bounds check, missing
  synchronization, races, uncoalesced access, changed math.

Being wrong is fine. Not predicting is not.

### Phase 4 — VERIFY (you)

The AI does not certify its own work. The baseline binary already contains a
CPU reference and prints `RESULT: PASS/FAIL` with a max-abs-error vs that
reference. Your variant must reproduce that gate.

Run your variant. Then **deliberately stress it**: run an event count that is
**not** a multiple of the block size (this is where dropped bounds checks and
off-by-one grid math show up). A variant that only passes on round numbers is
broken.

In [ ]:
!./mva_v1 20000000     # round number
!./mva_v1 19999999     # NOT a multiple of the block size — bounds-check trap
!./mva_v1 1000         # tiny workload

Record PASS/FAIL and max abs error for **each** run in your workbook. If any
run says `FAIL`, the optimization does not count until you fix it — go back to
Phase 2 and tell the AI exactly which case broke.

### Phase 5 — DIAGNOSE (you)

Now measure and explain. Profile the variant. If Nsight Systems is available on
the lab machine, capture a timeline; otherwise use the built-in timing and the
`nvprof`/`ncu` summary below.

In your workbook, answer:

- measured throughput vs your Phase-3 prediction — did it match? If not, why
  were you wrong? (This is the most valuable sentence in the whole module.)
- profiler evidence: achieved occupancy, memory throughput, stall reasons,
  transfer time vs compute time,
- the bottleneck **now**,
- the next optimization and which course concept it comes from.

In [ ]:
# Nsight Systems timeline (if available). Open the .qdrep/.nsys-rep in the GUI,
# or read the CLI summary. Look at H2D/D2H vs kernel time.
!nsys profile --stats=true -o mva_v1_profile ./mva_v1 20000000 || echo 'nsys not available — use the built-in timing and ncu below'

In [ ]:
# Nsight Compute kernel-level metrics: achieved occupancy and memory throughput.
!ncu --set basic --launch-count 1 ./mva_v1 1000000 || echo 'ncu not available — use nsys and the built-in timing'

## Repeat

Go back to Phase 1 with your next optimization. Log every iteration in the
workbook table (version, change, throughput, PASS?, bottleneck found). Aim for
at least two correct iterations with a profiler-backed reason for each.

## Before you submit

Run [grading/self_check.md](./grading/self_check.md) on yourself. The final box
is the one that matters: *if the AI transcript were deleted, would your workbook
still show that you understood what happened and why?* If not, you used the AI
as a crutch, not a tool.